# 02. Perturbaciones: grilla y justificación (T22, T23)

Cada perturbación de `src/perturbations.py` (T20) en sus tres niveles, sobre una imagen normal de Bremen.

Las perturbaciones se aplican **después de reducir a 640×360**, el tamaño del clásico. En la ubicación 2 las originales son 4608×2592: perturbarlas a ese tamaño y luego reducir promedia el ruido y borra la compresión JPEG.

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import perturbations as P
from src.features import HEIGHT, WIDTH

IMAGE = ROOT / 'data' / 'raw' / 'bremen' / 'Location1_Normal' / 'image_2024-07-16_19-47-11.jpg'
FIGURES = ROOT / 'results' / 'figuras'

original = cv2.resize(cv2.imread(str(IMAGE)), (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)
original.shape

## 1. Grilla perturbación × nivel (T22)

In [ ]:
names = list(P.PERTURBATIONS)
fig, axes = plt.subplots(len(names), 4, figsize=(12, 2.2 * len(names)))
for row, name in enumerate(names):
    axes[row, 0].imshow(original[..., ::-1])
    axes[row, 0].set_ylabel(name, rotation=0, ha='right', va='center')
    for col, level in enumerate(P.LEVELS, start=1):
        out = P.apply(name, original, level, seed=0)
        axes[row, col].imshow(out[..., ::-1])
        axes[row, col].set_title(f'nivel {level}', fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_title('original', fontsize=9)
fig.tight_layout()
FIGURES.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES / 'perturbaciones_grilla.png', dpi=110)
plt.show()

## 2. Justificación de los niveles (T23)

Cada nivel simula una situación real. Los valores son de diseño (D7), no medidos en campo; el conjunto de iluminación física de T81 es el que los valida.

| Perturbación | Nivel 1 | Nivel 2 | Nivel 3 | Situación real que simula |
|---|---|---|---|---|
| brillo oscuro | ×0,7 | ×0,5 | ×0,3 | Atardecer, interior con luz tenue, noche con alumbrado público |
| brillo claro | ×1,3 | ×1,6 | ×2,0 | Contraluz de ventana, sol directo sobre la lente |
| gamma | 1,5 | 2,0 | 3,0 | Subexposición no lineal del sensor (respuesta gamma distinta) |
| ruido gaussiano | σ 5 | σ 15 | σ 30 | Sensor con ganancia alta en poca luz (ISO alto) |
| sal y pimienta | 1 % | 5 % | 10 % | Píxeles defectuosos o errores de transmisión |
| JPEG | calidad 60 | calidad 30 | calidad 10 | Transmisión comprimida por red o celular con compresión fuerte |

**Por qué estos tres niveles:** el nivel 1 es una degradación leve y frecuente; el nivel 3 es el extremo de cada situación. La auditoría usa los tres (T70: test × 6 × 3). Hallazgo de la grilla: a 640×360 el nivel 1 del JPEG (calidad 60) y el del ruido gaussiano (σ 5) casi no se distinguen de la original. Si los descriptores tampoco cambian con ellos, es un resultado de la intensidad elegida y se reporta como tal. Si el nivel 3 resulta irreal en una cámara fija, se documenta como caso extremo; el conjunto de iluminación física (T75) valida la realidad de los niveles.